# Embed Compass Task 1 (Biome) and visualize with PCA + t-SNE

This notebook:
1. Loads a pretrained Waypoint model from the Hugging Face Hub.
2. Loads the first Compass benchmark task (`mgnify-biomes`) and concatenates **all** splits (train + validation + test).
3. Embeds each sample with the model (mirrors `embed.py`).
4. Plots PCA and t-SNE of the embeddings, colored by `Biome 1`, using Plotly.

Both **outpost-bio/Waypoint-6m** and **outpost-bio/Compass** are gated on the Hub. Request access and run `huggingface-cli login` (or set `HF_TOKEN`) before running this notebook.

In [ ]:
import sys
from pathlib import Path

# Make the repo importable when running from examples/.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel

import plotly.express as px

from embed import tokenize_for_embedding
from src.dataset import try_load_token_std_means
from src.models import _pool
from src.tokenizer import load_tokenizer

In [ ]:
MODEL_ID = "outpost-bio/Waypoint-170m"
COMPASS_REPO = "outpost-bio/Compass"
TASK_CONFIG = "mgnify-biomes"  # Compass task 1
COLOR_BY = "Biome 1"            # top-level biome label

POOLING = "last_token"
MAX_LENGTH = 512
BATCH_SIZE = 32

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

# Figure output dir + filename tag (so running multiple models doesn't overwrite).
MODEL_TAG = MODEL_ID.rsplit("/", 1)[-1]  # e.g. 'Waypoint-6m'
FIG_DIR = REPO_ROOT / "figs"
FIG_DIR.mkdir(exist_ok=True)

def save_fig(fig, name: str) -> Path:
    """Write a responsive standalone HTML. Filename is tagged with MODEL_TAG."""
    out = FIG_DIR / f"{name}__{MODEL_TAG}.html"
    fig.write_html(
        out,
        include_plotlyjs="cdn",
        full_html=True,
        config={"responsive": True},
    )
    print(f"  saved {out.relative_to(REPO_ROOT)}")
    return out


## 1. Load the Compass task 1 split

In [ ]:
ds = load_dataset(COMPASS_REPO, TASK_CONFIG)
split_dfs = []
for split_name in ("train", "validation", "test"):
    if split_name not in ds:
        continue
    sdf = ds[split_name].to_pandas()
    # The dataset already carries a 'Split' column, but make sure it's set
    # in case a config omits it.
    if "Split" not in sdf.columns:
        sdf["Split"] = split_name
    split_dfs.append(sdf)
df = pd.concat(split_dfs, ignore_index=True)
print(f"Loaded {len(df):,} samples across splits {df['Split'].value_counts().to_dict()}")
df.head(3)

## 2. Load the model + tokenizer

In [ ]:
tokenizer = load_tokenizer(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval().to(DEVICE)

token_std_means = try_load_token_std_means(MODEL_ID)
if token_std_means is not None:
    print(f"Loaded token_std_means ({len(token_std_means)} tokens) -- using z-score ordering")
else:
    print("No token_std_means found -- falling back to descending abundance order")

## 3. Tokenize and embed every sample

In [ ]:
samples = tokenize_for_embedding(df, tokenizer, MAX_LENGTH, token_std_means)
print(f"Tokenized {len(samples)} samples (seq length = {MAX_LENGTH})")

In [ ]:
loader = DataLoader(samples, batch_size=BATCH_SIZE, shuffle=False)
all_embeddings = []
with torch.no_grad():
    for batch in tqdm(loader, desc="Embedding", unit="batch"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        hidden = model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pooled = _pool(hidden, attention_mask, POOLING)
        all_embeddings.append(pooled.cpu())
embeddings = torch.cat(all_embeddings, dim=0).numpy()
print(f"Embeddings shape: {embeddings.shape}")

## 4. PCA and t-SNE
We run PCA on the raw embeddings and use a PCA-reduced version as the t-SNE input (standard practice — faster and de-noises the high-dim space).

In [ ]:
pca = PCA(n_components=2, random_state=0)
pca_coords = pca.fit_transform(embeddings)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")

n_pca_for_tsne = min(50, embeddings.shape[1], embeddings.shape[0])
tsne_input = PCA(n_components=n_pca_for_tsne, random_state=0).fit_transform(embeddings)
perplexity = float(min(30, max(5, (len(tsne_input) - 1) // 3)))
tsne_coords = TSNE(
    n_components=2,
    perplexity=perplexity,
    init="pca",
    learning_rate="auto",
    random_state=0,
).fit_transform(tsne_input)

## 6. Iterate over every biome rank

`mgnify-biomes` exposes 5 biome label columns (`Biome 1` … `Biome 5`) of increasing granularity. The cell below loops over them and plots PCA + t-SNE for each, skipping ranks where every value is missing.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

BIOME_COLS = [f"Biome {i}" for i in range(1, 6)]

for col in BIOME_COLS:
    if col not in df.columns:
        continue
    labels = df[col].fillna("(missing)").astype(str)
    if labels.nunique() <= 1 and labels.iloc[0] == "(missing)":
        print(f"Skipping {col} -- no labels present")
        continue

    frame = pd.DataFrame({
        "PCA1": pca_coords[:, 0],
        "PCA2": pca_coords[:, 1],
        "tSNE1": tsne_coords[:, 0],
        "tSNE2": tsne_coords[:, 1],
        col: labels.values,
    })

    pca_fig = px.scatter(frame, x="PCA1", y="PCA2", color=col, opacity=0.8)
    tsne_fig = px.scatter(frame, x="tSNE1", y="tSNE2", color=col, opacity=0.8)

    fig = make_subplots(rows=1, cols=2, subplot_titles=("PCA", "t-SNE"))
    seen = set()
    for src_fig, col_idx in ((pca_fig, 1), (tsne_fig, 2)):
        for trace in src_fig.data:
            trace.marker.size = 6
            # Share the legend across the two subplots: show each label only once.
            trace.showlegend = trace.name not in seen
            seen.add(trace.name)
            trace.legendgroup = trace.name
            fig.add_trace(trace, row=1, col=col_idx)
    fig.update_layout(
        title=f"{col} -- {labels.nunique()} unique labels ({len(frame):,} samples)",
        legend_title=col,
    )
    fig.update_xaxes(title_text="PCA1", row=1, col=1)
    fig.update_yaxes(title_text="PCA2", row=1, col=1)
    fig.update_xaxes(title_text="tSNE1", row=1, col=2)
    fig.update_yaxes(title_text="tSNE2", row=1, col=2)
    save_fig(fig, f"pca_tsne_{col.replace(' ', '_')}")
    fig.show()